# Silver Layer: Data Cleaning & Transformation
Processes bronze layer CSV files through PySpark for cleaning, deduplication, and type casting.

In [97]:
import sys
from pathlib import Path

work_dir = Path("/home/jovyan/work")
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os
import logging

# Initialize Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-silver") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
logger = logging.getLogger(__name__)

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


## 2. Setup and Initialization

In [98]:
# Paths
BRONZE_DIR = Path("/home/jovyan/work/output/bronze")
SILVER_DIR = Path("/home/jovyan/work/output/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Bronze directory: {BRONZE_DIR}")
print(f"Silver directory: {SILVER_DIR}")

# List bronze files
bronze_files = sorted(BRONZE_DIR.glob("*.csv"))
print(f"\nFound {len(bronze_files)} bronze files:")
for f in bronze_files:
    print(f"  - {f.name}")

Bronze directory: /home/jovyan/work/output/bronze
Silver directory: /home/jovyan/work/output/silver

Found 6 bronze files:
  - customers.csv
  - order_items.csv
  - orders.csv
  - payments.csv
  - products.csv
  - sellers.csv


## 3. Define Data Cleaning Functions

In [99]:
def clean_orders(df):
    """Clean and type orders data."""
    return df \
        .withColumn("order_id", col("order_id").cast(StringType())) \
        .withColumn("customer_id", col("customer_id").cast(StringType())) \
        .withColumn("order_status", col("order_status").cast(StringType())) \
        .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
        .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
        .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
        .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
        .withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date"))) \
        .dropDuplicates(["order_id"]) \
        .filter(col("order_id").isNotNull())

def clean_order_items(df):
    """Clean and type order_items data."""
    return df \
        .withColumn("order_id", col("order_id").cast(StringType())) \
        .withColumn("order_item_id", col("order_item_id").cast(IntegerType())) \
        .withColumn("product_id", col("product_id").cast(StringType())) \
        .withColumn("seller_id", col("seller_id").cast(StringType())) \
        .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
        .withColumn("price", col("price").cast(DecimalType(10, 2))) \
        .withColumn("freight_value", col("freight_value").cast(DecimalType(10, 2))) \
        .dropDuplicates(["order_id", "order_item_id"]) \
        .filter(col("order_id").isNotNull() & col("order_item_id").isNotNull())

def clean_customers(df):
    """Clean and type customers data."""
    return df \
        .withColumn("customer_id", col("customer_id").cast(StringType())) \
        .withColumn("customer_unique_id", col("customer_unique_id").cast(StringType())) \
        .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast(StringType())) \
        .withColumn("customer_city", col("customer_city").cast(StringType())) \
        .withColumn("customer_state", col("customer_state").cast(StringType())) \
        .dropDuplicates(["customer_id"]) \
        .filter(col("customer_id").isNotNull())

def clean_products(df):
    """Clean and type products data. Note: API has typo in column names (lenght vs length)."""
    return df \
        .withColumn("product_id", col("product_id").cast(StringType())) \
        .withColumn("product_category_name", coalesce(col("product_category_name"), lit("unknown"))) \
        .withColumn("product_name_lenght", col("product_name_lenght").cast(IntegerType())) \
        .withColumn("product_description_lenght", col("product_description_lenght").cast(IntegerType())) \
        .withColumn("product_photos_qty", col("product_photos_qty").cast(IntegerType())) \
        .withColumn("product_weight_g", col("product_weight_g").cast(DecimalType(10, 2))) \
        .withColumn("product_length_cm", col("product_length_cm").cast(DecimalType(10, 2))) \
        .withColumn("product_height_cm", col("product_height_cm").cast(DecimalType(10, 2))) \
        .withColumn("product_width_cm", col("product_width_cm").cast(DecimalType(10, 2))) \
        .dropDuplicates(["product_id"]) \
        .filter(col("product_id").isNotNull())

def clean_sellers(df):
    """Clean and type sellers data."""
    return df \
        .withColumn("seller_id", col("seller_id").cast(StringType())) \
        .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast(StringType())) \
        .withColumn("seller_city", col("seller_city").cast(StringType())) \
        .withColumn("seller_state", col("seller_state").cast(StringType())) \
        .dropDuplicates(["seller_id"]) \
        .filter(col("seller_id").isNotNull())

def clean_payments(df):
    """Clean and type payments data."""
    return df \
        .withColumn("order_id", col("order_id").cast(StringType())) \
        .withColumn("payment_sequential", col("payment_sequential").cast(IntegerType())) \
        .withColumn("payment_type", col("payment_type").cast(StringType())) \
        .withColumn("payment_installments", col("payment_installments").cast(IntegerType())) \
        .withColumn("payment_value", col("payment_value").cast(DecimalType(10, 2))) \
        .dropDuplicates(["order_id", "payment_sequential"]) \
        .filter(col("order_id").isNotNull() & col("payment_sequential").isNotNull())

print("Cleaning functions defined")

Cleaning functions defined


## 4. Process and Transform All Tables

In [100]:
# Process each bronze table
tables = {
    "orders": clean_orders,
    "order_items": clean_order_items,
    "customers": clean_customers,
    "products": clean_products,
    "sellers": clean_sellers,
    "payments": clean_payments,
}

silver_tables = {}

for table_name, clean_func in tables.items():
    bronze_file = BRONZE_DIR / f"{table_name}.csv"
    
    if not bronze_file.exists():
        print(f"  {table_name}: Bronze file not found")
        continue
    
    try:
        # Read bronze CSV
        df = spark.read.csv(str(bronze_file), header=True, inferSchema=True)
        original_count = df.count()
        
        # Apply cleaning
        df_clean = clean_func(df)
        clean_count = df_clean.count()
        
        # Save to silver
        silver_file = SILVER_DIR / f"{table_name}.csv"
        df_clean.coalesce(1).write.csv(str(silver_file), header=True, mode="overwrite")
        
        silver_tables[table_name] = df_clean
        
        print(f"✓ {table_name:15s}: {original_count:8d} → {clean_count:8d} rows")
    except Exception as e:
        print(f"✗ {table_name:15s}: {str(e)}")

print(f"\nProcessed {len(silver_tables)} tables")

✓ orders         :     1044 →     1000 rows
✓ order_items    :     3000 →     1000 rows
✓ customers      :     3000 →     1000 rows
✓ products       :     3000 →     1000 rows
✓ sellers        :     3000 →     1000 rows
✓ payments       :     3000 →     1000 rows

Processed 6 tables


## 5. Silver Layer Schema Verification

In [101]:
# Verification: Show schema and sample for key tables
for table_name in ["orders", "customers", "products"]:
    if table_name in silver_tables:
        df = silver_tables[table_name]
        print(f"\n{'='*60}")
        print(f"TABLE: {table_name.upper()}")
        print(f"{'='*60}")
        print(f"Rows: {df.count()}")
        print(f"\nSchema:")
        df.printSchema()


TABLE: ORDERS
Rows: 1000

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: date (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)


TABLE: CUSTOMERS
Rows: 1000

Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)


TABLE: PRODUCTS
Rows: 1000

Schema:
root
 |-- product_id: st

## 6. Data Quality Summary